In [ ]:
import torch
import torch.nn as nn
import pickle
import numpy as np, shutil
from torch.nn import functional as F
import time
import math
import os
import random

In [ ]:
#hyperparams

batch_size = 64
block_size = 256
max_iters = 35000
eval_interval = 200
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 50
n_embd = 512
n_head = 8
n_layer = 8
dropout = 0.0

torch.manual_seed(1337)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/Jamil_Karpathy_lessons"

for name in ["train.bin", "val.bin", "merges.pkl"]:
  shutil.copy(f"{DRIVE_DIR}/{name}", f"/content/{name}")

train_data = np.memmap("/content/train.bin", dtype = np.uint16, mode = "r")
val_data = np.memmap("/content/val.bin", dtype = np.uint16, mode = "r")

def build_vocab(merges):
  vocab = {i : bytes([i]) for i in range(256)}
  for (a, b), idx in merges.items():
    vocab[idx] = vocab[a] + vocab[b]
  return vocab

def decode(ids, vocab):
  return b"".join(vocab[i] for i in ids).decode("utf-8", errors = "replace")

merges = pickle.load(open("/content/merges.pkl", "rb"))
vocab = build_vocab(merges)
vocab[4096] = b"<|endoftext|>"

vocab_size = 4097
print(len(train_data), len(val_data))

255925379 4886831


In [ ]:
def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size - 1, (batch_size,))
  x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
  y = torch.stack([torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64)) for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

In [ ]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ["train", "val"]:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [ ]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)
    self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  #manual attention
  #def forward(self, x):
  #  B, T, C = x.shape
  #  k = self.key(x)
  #  q = self.query(x)

  #  wei = q @ k.transpose(-2, -1) * (key.shape[-1] ** -0.5)
  #  wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
  #  wei = F.softmax(wei, dim = -1)
  #  wei = self.dropout(wei)

  #  v = self.value(x)
  #  out = wei @ v
  #  return out

  def forward(self, x):
    B, T, C = x.shape
    k = self.key(x)
    v = self.value(x)
    q = self.query(x)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask = None, dropout_p = dropout if self.training else 0.0, is_causal = True)
    return out


In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(num_heads * head_size, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))
    return out

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, n_embd*4),
        nn.ReLU(),
        nn.Linear(n_embd*4, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
class Block(nn.Module):

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

In [ ]:
#vocab_size = 4097
padded_vocab_size = 4160        # 4097 rounded up to a multiple of 64 (4104 is the minimum multiple of 8)

In [ ]:
class GPTLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()

    #self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.token_embedding_table = nn.Embedding(padded_vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final LN
    self.lm_head = nn.Linear(n_embd, padded_vocab_size)

    self.token_embedding_table.weight = self.lm_head.weight   # <- weight tying    ### new ***

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

  def forward(self, idx, targets = None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device = idx.device)) # (T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) # (B, T, C)
    x = self.ln_f(x) # (B, T, C)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  #def forward(self, idx, targets = None):
  #  B, T = idx.shape

    #tok_emb = self.token_embedding_table(idx)
    #pos_emb = self.position_embedding_table(torch.arange(T, device = device))
    #x = tok_emb + pos_emb
    #x = self.blocks(x)
    #x = self.ln_f(x)                                  # (B, T, C)

    #if targets is None:
    #  return self.lm_head(x), None

    #if use_fused_ce:
    #  h = x.reshape(B*T, -1).to(torch.float16)        # ln_f outputs fp32 under autocast
    #  w = self.lm_head.weight.to(torch.float16)
    #  b = self.lm_head.bias.to(torch.float16)
    #  loss = fused_ce(w, h, targets.reshape(B*T), b)
    #  return None, loss

    #logits = self.lm_head(x)
    #loss = F.cross_entropy(logits.view(B*T, -1), targets.reshape(B*T))
    #return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :vocab_size]
      probs = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx

In [ ]:
!pip install nvidia-ml-py   # run once in a separate cell if the import fails

In [ ]:
import pynvml
pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)

def gpu_stats():
  clk = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_SM)
  temp = pynvml.nvmlDeviceGetTemperature(h, pynvml.NVML_TEMPERATURE_GPU)
  pwr = pynvml.nvmlDeviceGetPowerUsage(h) / 1000
  try:
    r = pynvml.nvmlDeviceGetCurrentClocksThrottleReasons(h)
  except Exception:
    r = 0
  reasons = []
  if r & 0x4:  reasons.append("power-cap")
  if r & 0x8:  reasons.append("hw-slowdown")
  if r & 0x20: reasons.append("thermal-sw")
  if r & 0x40: reasons.append("thermal-hw")
  if r & 0x80: reasons.append("power-brake")
  return f"sm {clk} MHz, {temp} C, {pwr:.0f} W, throttle: {','.join(reasons) or 'none'}"

In [ ]:
raw_model = GPTLanguageModel().to(device)
model = torch.compile(raw_model, mode = "reduce-overhead")

In [ ]:
print(raw_model.token_embedding_table.weight is raw_model.lm_head.weight)   # True

True


In [ ]:
# -----------------------------------------
# 1. CONFIGS

#max_iters = 35_000
warmup_iters = 500
min_lr = learning_rate / 10

checkpoint_interval = 1000
sample_interval = 500

early_stopping = False
patience = 15
min_delta = 0.002
min_steps = 5000

SAVE_DIR = f"{DRIVE_DIR}/gpt_pretraining"
os.makedirs(SAVE_DIR, exist_ok=True)

LATEST_PATH = f"{SAVE_DIR}/latest_checkpoint.pt"
BEST_PATH = f"{SAVE_DIR}/best_pretrained_model.pt"
FINAL_PATH = f"{SAVE_DIR}/final_pretrained_model.pt"

# Set True ONLY when resuming a saved run.
RESUME = False

# Preserve your original optimizer configuration.
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=learning_rate, fused=True)

scaler = torch.amp.GradScaler("cuda")

start_iter = 0
best_val_loss = float("inf")
bad_evals = 0
last_val_loss = None


# -------------------------------------------
# 2. LEARNING RATE SCHEDULE


def get_lr(it):

    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters

    if it >= max_iters:
        return min_lr

    progress = ((it - warmup_iters) /(max_iters - warmup_iters))
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + coeff * (learning_rate - min_lr)


# --------------------------------------------
# 3. CHECKPOINT HELPERS

def atomic_save(obj, path):

    temp_path = path + ".tmp"

    torch.save(obj, temp_path)
    os.replace(temp_path, path)


def save_checkpoint(step):

    checkpoint = {
        "step": step,
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),

        "best_val_loss": best_val_loss,
        "bad_evals": bad_evals,
        "last_val_loss": last_val_loss,

        "learning_rate": learning_rate,
        "max_iters": max_iters,
        "warmup_iters": warmup_iters,
        "min_lr": min_lr,

        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        #"numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
    }

    atomic_save(checkpoint, LATEST_PATH)

    print(f"Checkpoint saved at step {step}")


def save_model(path, step, val_loss):

    artifact = {
        "model": raw_model.state_dict(),
        "step": step,
        "val_loss": val_loss,

        # Tokenizer vocabulary: 0 to 4096
        "tokenizer_vocab_size": 4097,
        "padded_vocab_size": padded_vocab_size,

        # Model architecture
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
    }

    atomic_save(artifact, path)


# -----------------------------------------------
# 4. OPTIONAL RESUME


if RESUME:

    checkpoint = torch.load(LATEST_PATH, map_location=device,weights_only=True)

    assert checkpoint["max_iters"] == max_iters
    assert checkpoint["warmup_iters"] == warmup_iters
    assert checkpoint["min_lr"] == min_lr
    assert checkpoint["learning_rate"] == learning_rate

    raw_model.load_state_dict(checkpoint["model"])

    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])

    start_iter = checkpoint["step"]
    best_val_loss = checkpoint["best_val_loss"]
    bad_evals = checkpoint["bad_evals"]
    last_val_loss = checkpoint["last_val_loss"]

    torch.set_rng_state(checkpoint["torch_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    #np.random.set_state(checkpoint["numpy_rng"])
    random.setstate(checkpoint["python_rng"])

    print(f"Resuming from step {start_iter}")


# ------------------------------------------
# 5. TEXT GENERATION

def sample_text(n_tokens=300):

    was_training = raw_model.training
    raw_model.eval()

    context = torch.full((1, 1),4096,dtype=torch.long,device=device)

    try:
        with torch.inference_mode(), torch.autocast(device_type="cuda",dtype=torch.float16):

            output = raw_model.generate(context, max_new_tokens=n_tokens)[0].tolist()

        return decode(output, vocab)

    finally:
        raw_model.train(was_training)


# ---------------------------------------
# 6. EVALUATION

step_times = []

def evaluate_and_save(step):

    global best_val_loss, bad_evals, last_val_loss

    losses = estimate_loss()

    train_loss = float(losses["train"])
    val_loss = float(losses["val"])

    last_val_loss = val_loss
    avg_time = (sum(step_times) / len(step_times) if step_times else 0)
    tokens_per_step = batch_size * block_size
    tokens_per_sec = (tokens_per_step / avg_time if avg_time > 0 else 0)
    peak_mem = (torch.cuda.max_memory_allocated() / 1e9)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"step {step}: "
        f"train loss {train_loss:.4f}, "
        f"val loss {val_loss:.4f}, "
        f"lr {current_lr:.2e}, "
        f"avg step time {avg_time * 1000:.2f} ms, "
        f"tokens/step {tokens_per_step:,}, "
        f"tokens/sec {tokens_per_sec:,.0f}, "
        f"peak mem {peak_mem:.2f} GB, "
        f"{gpu_stats()}"
    )

    improvement = best_val_loss - val_loss

    # Always preserve the lowest observed val loss.
    if improvement > 0:

        best_val_loss = val_loss
        save_model(BEST_PATH, step, val_loss)
        print(f"Best model updated: {val_loss:.4f}")

    # Only relevant when early stopping is enabled.
    if improvement >= min_delta:
        bad_evals = 0
    else:
        bad_evals += 1

    return val_loss


# ------------------------------------
# 7. TRAINING LOOP

torch.cuda.reset_peak_memory_stats()

completed_steps = start_iter
stopped_early = False
interrupted = False

try:

    for it in range(start_iter, max_iters):

        lr = get_lr(it)
        for group in optimizer.param_groups:
            group["lr"] = lr

        # Training batch
        xb, yb = get_batch("train")

        torch.cuda.synchronize()
        start = time.perf_counter()

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits, loss = model(xb, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        torch.cuda.synchronize()
        step_times.append(time.perf_counter() - start)

        # One training iteration completed.
        completed_steps = it + 1

        # Evaluation
        if (completed_steps % eval_interval == 0 or completed_steps == max_iters):
            evaluate_and_save(completed_steps)
            step_times = []
            if (early_stopping and completed_steps >= min_steps and bad_evals >= patience):
                print(f"\nEarly stopping at step" f"{completed_steps}.")
                stopped_early = True

        # Sample generation
        if (completed_steps % sample_interval == 0 and not stopped_early):
            print(
                f"\n--- sample at step "
                f"{completed_steps} ---\n"
                f"{sample_text()}\n"
                f"--- end sample ---\n"
            )

        # Rolling checkpoint
        # ----------------------------------

        if completed_steps % checkpoint_interval == 0:
            save_checkpoint(completed_steps)

            # Check checkpoint folder size
            print("---------------------------")
            os.system(f'du -sh "{SAVE_DIR}"')
            print("---------------------------")

        if stopped_early:
            break

except KeyboardInterrupt:
    interrupted = True

    print(
        f"\nTraining interrupted at step {completed_steps}."
        f"\nResume from the last periodic checkpoint."
    )


# -------------------------------------------
# 8. FINAL SAVE

if (
    not interrupted
    and completed_steps % eval_interval != 0
):

    evaluate_and_save(completed_steps)

# Save the latest training state,
# including optimizer and scaler.
if not interrupted:
    save_checkpoint(completed_steps)

# Only label a model as final if the
# requested run actually finished
# or was stopped deliberately.
if not interrupted:

    save_model(FINAL_PATH, completed_steps, last_val_loss)

print("\nTraining ended.")
print(f"Completed steps: {completed_steps}")
print(f"Best validation loss: {best_val_loss:.4f}")

if not interrupted:

    print(
        f"\n--- final sample ---\n"
        f"{sample_text(500)}\n"
        f"--- end sample ---"
    )

print("\nSaved files:")
print(LATEST_PATH)
print(BEST_PATH)

if not interrupted:
    print(FINAL_PATH)

W0922 11:53:15.902000 553 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step 200: train loss 4.5365, val loss 4.5328, lr 1.20e-04, avg step time 807.75 ms, tokens/step 16,384, tokens/sec 20,283, peak mem 4.22 GB, sm 840 MHz, 77 C, 69 W, throttle: power-cap
Best model updated: 4.5328
step 400: train loss 3.5629, val loss 3.5558, lr 2.40e-04, avg step time 305.36 ms, tokens/step 16,384, tokens/sec 53,655, peak mem 4.22 GB, sm 720 MHz, 76 C, 68 W, throttle: power-cap
Best model updated: 3.5558

--- sample at step 500 ---
<|endoftext|>Once upon a time, there was a little girl named Shig ostrich and humble sple, but when he saw what was so coffee. Jill was walking in an old room. One day, he accidentally had a big coin in the swing.

Tommy went to his river and saw all the sun, can keep right. So, he said Benny, "No,ject!"<|endoftext|>Once upon a time there was a othershery in the bucket under grumpy house. Oo', the world around his yellow and went as enjoy what was not kept, when they came and made it wid on it. Then, they reached the ground to jumps away. The

In [ ]:
import regex as re, collections

# split into word-ish chunks so merges never cross word boundaries
pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

def merge(ids, pair, idx):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            out.append(idx); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def train_bpe(text, vocab_size):
    counts = collections.Counter(pat.findall(text))
    words = {w: list(w.encode("utf-8")) for w in counts}
    merges = {}
    for idx in range(256, vocab_size):
        stats = collections.Counter()
        for w, ids in words.items():
            c = counts[w]
            for p in zip(ids, ids[1:]):
                stats[p] += c
        if not stats: break
        pair = max(stats, key=stats.get)
        merges[pair] = idx
        for w in words:
            words[w] = merge(words[w], pair, idx)
    return merges

def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), idx in merges.items():   # dicts keep insertion order
        vocab[idx] = vocab[a] + vocab[b]
    return vocab

def encode(text, merges, cache):
    out = []
    for w in pat.findall(text):
        if w not in cache:
            ids = list(w.encode("utf-8"))
            while len(ids) >= 2:
                pairs = set(zip(ids, ids[1:]))
                pair = min(pairs, key=lambda p: merges.get(p, float("inf")))
                if pair not in merges: break
                ids = merge(ids, pair, merges[pair])
            cache[w] = ids
        out.extend(cache[w])
    return out


In [ ]:
#import torch
#device = "cuda" if torch.cuda.is_available() else "cpu"
import regex as re, collections

BEST_PATH = "/content/drive/MyDrive/Jamil_Karpathy_lessons/gpt_pretraining/best_pretrained_model.pt"

# Recreate model
raw_model = GPTLanguageModel().to(device)

# Load best pretrained weights
checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=True
)

raw_model.load_state_dict(checkpoint["model"])
raw_model.eval()

print(
    f"Loaded checkpoint from step {checkpoint['step']}, "
    f"val loss = {checkpoint['val_loss']:.4f}"
)

# -----------------------------------
# Prompt and generate
# -----------------------------------

cache = {}
pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
prompt = "Once upon a time, there was"

prompt_ids = encode(prompt, merges, cache)

x = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)

with torch.inference_mode():
    out = raw_model.generate(
        x,
        max_new_tokens=500
    )[0].tolist()

print(decode(out, vocab))

Loaded checkpoint from step 34800, val loss = 1.4173
Once upon a time, there was a beautiful diary. In it all sorts of lovely colours, and it was very special. One day, the diary started to make its secret hiding out. It chose itself to hide in a secret place, so nobody could find it. But the secret secret stayed otherwise, hiding away! The diary was just the spot and that it was able to hide. In the secret place, the diary was the perfect place for the diary! And so very soon, the diary was the star of the diary! The diary was happy to be safe, and the secret place was never so close.<|endoftext|>A rabbit and a fox were in the woods. The rabbit was feeling brave and decided to take a look around. He saw something amazing and was very scared. It was a huge snake, and it was running away. The rabbit was very worried so he ran as fast as he could. He zigzagged and zipped along the path, trying to escape from the snake. But it was too late. The snake caught up to him and it ate him up. Th